# Colab Run & Implementation Template

This notebook contains cells to: clone the repository, install Colab-specific dependencies, set ngrok and service tokens, run the Colab summary service, and basic project scaffolding examples.


## 0) CS6493 GCP Cost & Region Reminder

> Follow this checklist before starting runtime to avoid burning credits.

- Use template: `template-g2-standard-16-L4`
- Region must be: `asia-east1`
- Do not create/use runtimes in `us-east1` or `us-central1` (especially V100-related configs are expensive).
- Delete unnecessary templates/runtimes in other regions.
- Manually stop runtime after each session, even if idle auto-shutdown exists.

If your current runtime is not in `asia-east1`, stop it and recreate with the required template first.

## 1) Environment setup (Colab + branch)

Run this code in a Colab code cell to clone and switch to the current branch `sunbaozheng`.

```bash
# Clone repository
!git clone https://github.com/Yks-ydl/smart_meeting_assistant.git repo

# Switch to target branch
%cd repo
!git fetch --all
!git checkout sunbaozheng

# Enter project and install dependencies
%cd smart_meeting_assistant
!pip install -r scripts/colab/requirements-colab.txt
```

Tip: if `git checkout sunbaozheng` fails, run `!git branch -a` to verify branch visibility and `!git fetch origin sunbaozheng` first.

In [ ]:
# Execute setup: clone repo, switch branch, install dependencies
!git clone https://github.com/Yks-ydl/smart_meeting_assistant.git repo
%cd repo
!git fetch --all
!git checkout sunbaozheng
%cd smart_meeting_assistant
!pip install -r scripts/colab/requirements-colab.txt


## 2) Set ngrok token and generate secure service token

Never hardcode secrets in notebook files. Use placeholders or Colab Secrets.

`REPLACE_WITH_SECURE_TOKEN` should be a random application token shared by:
- Colab runtime env: `SUMMARY_REMOTE_AUTH`
- local gateway env: `SUMMARY_REMOTE_AUTH_TOKEN`

```python
import os
import secrets
from pyngrok import ngrok

# 1) Set your ngrok account token (from ngrok dashboard)
os.environ['NGROK_AUTHTOKEN'] = 'YOUR_NGROK_AUTHTOKEN'

# 2) Generate secure token for summary API auth
secure_token = secrets.token_urlsafe(32)
os.environ['SUMMARY_REMOTE_AUTH'] = secure_token
print('Generated SUMMARY_REMOTE_AUTH token:')
print(secure_token)

# 3) Start tunnel
ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
public_url = ngrok.connect(8000).public_url
print('Public URL:', public_url)
```

Security note: if any real token was ever committed into notebook history, rotate it immediately.

In [ ]:
# Execute auth setup: set ngrok token, generate secure summary token, start tunnel
import os
import secrets
from pyngrok import ngrok

# Replace with your own ngrok token from dashboard
os.environ['NGROK_AUTHTOKEN'] = 'YOUR_NGROK_AUTHTOKEN'

if os.environ['NGROK_AUTHTOKEN'] == 'YOUR_NGROK_AUTHTOKEN':
    raise ValueError('Please replace YOUR_NGROK_AUTHTOKEN before running this cell.')

secure_token = secrets.token_urlsafe(32)
os.environ['SUMMARY_REMOTE_AUTH'] = secure_token
print('Generated SUMMARY_REMOTE_AUTH token (copy to local .env as SUMMARY_REMOTE_AUTH_TOKEN):')
print(secure_token)

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
public_url = ngrok.connect(8000).public_url
print('Public URL:', public_url)


In [ ]:
# Optional executable helper cell: print local .env values
import os

if 'public_url' not in globals() or not public_url:
    raise ValueError('Run the auth/tunnel setup cell first to create public_url.')

public_summary_url = f"{public_url}/api/v1/summary/generate"
print('Set these values in local .env:')
print('SUMMARY_EXECUTION_MODE=remote')
print(f'SUMMARY_SERVICE_URL={public_summary_url}')
print(f"SUMMARY_REMOTE_AUTH_TOKEN={os.environ.get('SUMMARY_REMOTE_AUTH', '')}")


## 3) Start the FastAPI Colab service

Run the service entrypoint which loads the model and starts `uvicorn`. The `colab_entry.py` included in `scripts/colab/` will start the FastAPI app and print the public URL obtained from ngrok.

```bash
# In Colab (bash cell)
# Adjust python path if necessary
!python -u scripts/colab/colab_entry.py
```

If using the notebook, prefer running it as a background process and check logs: `nohup python -u scripts/colab/colab_entry.py &`

In [ ]:
# Execute service startup (foreground)
!python -u scripts/colab/colab_entry.py


In [ ]:
## 4) Project structure and minimal boilerplate

Create minimal folders and files used by the service. These cells write simple placeholders so you can iterate quickly.

```python
from pathlib import Path
proj = Path('.')
src = proj / 'src'
tests = proj / 'tests'
src.mkdir(exist_ok=True)
tests.mkdir(exist_ok=True)
# Minimal package init
pkg = src / 'summary_service'
pkg.mkdir(exist_ok=True)
(pkg / '__init__.py').write_text('# summary_service package')

# Write a tiny README
Path('README_COLAB.md').write_text('# Colab run notes\nUse this file for quick notes')
print('Created minimal structure under', proj)
```

In [ ]:
## 5) Implement a tiny core module (example)

Write a small summarizer wrapper module that can be used for local testing. In real usage, replace implementation with the transformer model loader in `colab_model_runtime.py`.

```python
core_code = '''"""Minimal summarizer wrapper for quick tests"""

def summarize_text(text: str) -> str:
    """Return a very short placeholder summary for testing."""
    if not text or not text.strip():
        return ''
    # naive placeholder: return first sentence
    return text.strip().split('.')[:1][0]
'''

Path('src/summary_service/core.py').write_text(core_code)
print('Wrote src/summary_service/core.py')
```

In [ ]:
## 6) Command-line entry point example

Create a simple CLI wrapper to call the `summarize_text` function.

```python
cli_code = '''import argparse
from summary_service.core import summarize_text

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--text', required=True)
    args = p.parse_args()
    print(summarize_text(args.text))
'''
Path('src/summary_service/cli.py').write_text(cli_code)
print('Wrote CLI at src/summary_service/cli.py')
```

In [ ]:
## 7) Unit tests (pytest)

Create simple pytest tests for the core summarizer.

```python
test_code = '''from summary_service.core import summarize_text


def test_empty():
    assert summarize_text('') == ''


def test_basic():
    assert summarize_text('Hello world. More text.') == 'Hello world'
'''
Path('tests/test_core.py').write_text(test_code)
print('Wrote tests/test_core.py')
```

## 8) Run tests and view output

Run tests from the notebook or shell. In Colab, use `!pytest -q`.

```bash
# Run full pytest suite
!pytest -q

# Run single test file
!pytest -q tests/test_core.py
```

In [ ]:
## 9) Logging and error handling

Add basic logging configuration to the core module and demonstrate raising/catching an error.

```python
logging_code = '''import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def example():
    logger.info('Example log')
    raise ValueError('Demonstration error')
'''
Path('src/summary_service/logging_demo.py').write_text(logging_code)
print('Wrote logging demo')
```

In [ ]:
## 10) Continuous integration (GitHub Actions)

Write a minimal workflow that installs Python and runs tests.

```python
workflow = '''name: Python CI
on: [push]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      - name: Install dependencies
        run: |
          pip install -r scripts/colab/requirements-colab.txt
      - name: Run tests
        run: pytest -q
'''
Path('.github/workflows/python-ci.yml').parent.mkdir(parents=True, exist_ok=True)
Path('.github/workflows/python-ci.yml').write_text(workflow)
print('Wrote .github/workflows/python-ci.yml')
```

In [ ]:
## 11) Packaging and distribution (build)

Create minimal `pyproject.toml` and show commands to build a wheel and sdist.

```python
pyproject = '''[build-system]
requires = ["setuptools>=61.0", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "smart_meeting_summary_test"
version = "0.0.1"
readme = "README_COLAB.md"
'''
Path('pyproject.toml').write_text(pyproject)
print('Wrote pyproject.toml')

# Commands to build (commented):
# !python -m pip install build
# !python -m build
```

In [ ]:
## 12) Notebook demo: import and run

Quick demo that imports the local package and runs the summarizer function.

```python
import sys
sys.path.insert(0, 'src')
from summary_service.core import summarize_text

print(summarize_text('This is a demo. It should return the first sentence.'))
```